# Unit 2: 神经网络基础

## 学习目标
- 理解 `nn.Module` 的核心设计
- 掌握线性层、激活函数、损失函数和优化器
- 用 PyTorch 搭建并训练第一个多层感知机 (MLP)
- 在 MNIST 数据集上达到 97%+ 准确率

## 2.1 nn.Module：一切网络的基类

PyTorch 中所有神经网络都继承自 `nn.Module`。它提供了：
- **参数管理**：自动追踪可学习参数（权重和偏置）
- **设备管理**：`.to(device)` 一键移动所有参数
- **序列化**：`state_dict()` 保存/加载模型

核心规则：
1. `__init__` 中定义网络层
2. `forward` 中定义前向传播逻辑
3. **永远不要直接调用 `forward()`**，使用 `model(x)` 即可

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

## 2.2 第一个 nn.Module 示例

fc 是 Fully Connected（全连接层）的缩写。  
PyTorch每层实际shape采用 (out_features, in_features)，配合 x @ W^T：  
- 前向：x (B, in) @ W^T (in, out) → 直接就是 (B, out)  
- 反向对x求导：grad_y (B, out) @ W (out, in) → 直接就是 (B, in)  
- 反向对W求导：grad_y^T (out, B) @ x (B, in) → 直接就是 (out, in)  

关键优势：反向传播时，权重的梯度形状和权重本身完全一致，不需要额外的转置操作。

In [ ]:
class MyFirstModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = MyFirstModel(input_dim=10, hidden_dim=20, output_dim=5)
print(model)
print(f"\n参数总数: {sum(p.numel() for p in model.parameters()):,}")

for name, param in model.named_parameters():
    print(f"  {name}: {param.shape}")

In [ ]:
x = torch.randn(3, 10)
y = model(x)
print(f"Input: {x.shape} -> Output: {y.shape}")

model.to(device)
x_gpu = torch.randn(3, 10).to(device)
y_gpu = model(x_gpu)
print(f"Model on {device}: {y_gpu.device}")

## 2.3 线性层 (nn.Linear)

$$\mathbf{y} = \mathbf{x}\mathbf{W}^T + \mathbf{b}$$

- `in_features`: 输入维度
- `out_features`: 输出维度
- `bias`: 是否使用偏置（默认 True）

权重初始化：默认使用 Kaiming Uniform。  
Kaiming均匀分布（也叫He初始化）是深度学习中一种重要的权重初始化方法，由何恺明（Kaiming He）在2015年提出。

In [ ]:
linear = nn.Linear(4, 3)
print(f"Weight shape: {linear.weight.shape}")
print(f"Bias shape: {linear.bias.shape}")
print(f"Weight:\n{linear.weight.data}")

x = torch.randn(2, 4)
y = linear(x)
print(f"\nInput {x.shape} -> Output {y.shape}")
print(f"Output:\n{y}")

print(f"\n手动验证: W @ x + b")
manual = x @ linear.weight.T + linear.bias
# 比较两个张量是否近似相等的函数
print(f"Match: {torch.allclose(y, manual)}")

## 2.4 激活函数

激活函数引入**非线性**，使网络能学习复杂模式。

| 函数 | 公式 | 特点 |
|------|------|------|
| **ReLU** | $\max(0, x)$ | 最常用，简单高效 |
| **Sigmoid** | $\frac{1}{1+e^{-x}}$ | 输出 [0,1]，易梯度消失 |
| **Tanh** | $\frac{e^x-e^{-x}}{e^x+e^{-x}}$ | 输出 [-1,1]，零中心 |
| **LeakyReLU** | $\max(0.01x, x)$ | 解决 ReLU 死亡问题 |
| **Softmax** | $\frac{e^{x_i}}{\sum e^{x_j}}$ | 多分类概率输出 |

In [ ]:
x = torch.linspace(-5, 5, 200)

activations = {
    "ReLU": F.relu(x),
    "Sigmoid": torch.sigmoid(x),
    "Tanh": torch.tanh(x),
    "LeakyReLU": F.leaky_relu(x, negative_slope=0.1),
}

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, (name, y) in zip(axes, activations.items()):
    # .numpy() 只能将 CPU 上的张量 转换为 NumPy 数组。
    ax.plot(x.numpy(), y.numpy())
    ax.set_title(name)
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color="gray", linewidth=0.5)
    ax.axvline(x=0, color="gray", linewidth=0.5)
plt.suptitle("Activation Functions", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
logits = torch.tensor([2.0, 1.0, 0.1])
probs = F.softmax(logits, dim=0)
print(f"Logits: {logits}")
print(f"Softmax probabilities: {probs}")
print(f"Sum = {probs.sum():.4f}")

## 2.5 损失函数

损失函数衡量模型预测与真实标签之间的差距。

In [ ]:

# 均方误差损失函数（Mean Squared Error Loss，简称 MSE Loss）
mse = nn.MSELoss()
pred = torch.tensor([1.0, 2.0, 3.0])
target = torch.tensor([1.5, 2.5, 2.5])
loss_mse = mse(pred, target)
print(f"MSE Loss: {loss_mse:.4f}")
print(f"手动: mean((pred-target)^2) = {((pred-target)**2).mean():.4f}")

# 二分类交叉熵损失函数（Binary CrossEntropyropy Loss，简称 BCE Loss）
# 判断一个东西“是”或“不是”，输出只有两种可能，分别对应类别0和类别1
bce = nn.BCEWithLogitsLoss()
logits_b = torch.randn(4)
labels_b = torch.tensor([1.0, 0.0, 1.0, 0.0])
loss_bce = bce(logits_b, labels_b)
print(f"\nBCEWithLogitsLoss: {loss_bce:.4f}")
print(f"用于二分类, 内部自动完成 Sigmoid + BCELoss")

# 交叉熵损失函数（CrossEntropy Loss，简称 CE Loss）
# 多分类问题，输出是一个类别概率分布，每个样本有多个类别标签
ce = nn.CrossEntropyLoss()
logits = torch.randn(3, 5) # 3个样本，5个类别
labels = torch.tensor([1, 0, 4]) # 3个样本，每个样本的真实类别标签
loss_ce = ce(logits, labels)
print(f"\nCrossEntropyLoss: {loss_ce:.4f}")
print(f"CrossEntropyLoss, 内部自动完成 LogSoftmax + NLLLoss")

## 2.6 优化器

优化器根据梯度更新模型参数。

| 优化器 | 特点 |
|--------|------|
| **SGD** | 基础，需要手动调学习率，常配合 momentum |
| **Adam** | 自适应学习率，最常用，收敛快 |
| **AdamW** | Adam + 解耦权重衰减，推荐首选 |

In [ ]:
model = nn.Linear(2, 1)

x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
y = torch.tensor([[5.0], [11.0]])

optimizer_sgd = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
optimizer_adam = optim.Adam(model.parameters(), lr=0.01)
optimizer_adamw = optim.AdamW(model.parameters(), lr=0.01, weight_decay=1e-4)

print("优化器使用模板:")
print("  optimizer.zero_grad()   # 清零梯度")
print("  loss.backward()         # 反向传播")
print("  optimizer.step()        # 更新参数")

criterion = nn.MSELoss()
pred = model(x)
loss = criterion(pred, y)
optimizer_adam.zero_grad()
loss.backward()
optimizer_adam.step()
print(f"\n初始 loss: {loss.item():.4f}")

## 2.7 学习率调度器

动态调整学习率可以加速收敛、提升最终精度。

In [ ]:
model = nn.Linear(10, 1)
optimizer = optim.SGD(model.parameters(), lr=1.0)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

lrs = []

for epoch in range(10):
    # 记录当前学习率
    lrs.append(optimizer.param_groups[0]["lr"])
    
    # 模拟训练步骤（需要先有optimizer.step()）
    optimizer.step() # 更新模型参数
    scheduler.step() # 按照预设的策略调整学习率

plt.plot(range(10), lrs, "o-")
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.title("StepLR (step_size=3, gamma=0.5)")
plt.grid(True, alpha=0.3)
plt.show()

## 2.8 实战：MLP 训练 MNIST

MLP 的全称是 Multilayer Perceptron（多层感知器）。

MNIST：28x28 灰度手写数字图片，10 个类别（0-9）。

模型架构：`784 -> 256 -> 128 -> 10`

In [ ]:
# 定义数据预处理流程
transform = transforms.Compose([
    # 第1步：将PIL图像或numpy数组转换为PyTorch张量，并将像素值从[0,255]缩放到[0,1]
    transforms.ToTensor(),
    # 第2步：对图像进行标准化处理，使用MNIST数据集的均值(0.1307)和标准差(0.3081)
    # 公式: output = (input - mean) / std，使数据分布更接近标准正态分布
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print(f"Training samples: {len(train_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")

images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}")
print(f"Label shape: {labels.shape}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap="gray")
    ax.set_title(f"Label: {labels[i].item()}")
    ax.axis("off")
plt.suptitle("MNIST Samples")
plt.show()

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = x.view(x.size(0), -1) # x.view(64, -1)	(64, 784) ← 因为 1×28×28 = 784
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

model = MLP().to(device)
print(model)
print(f"\n参数总量: {sum(p.numel() for p in model.parameters()):,}")

for name, param in model.named_parameters():
    print(f"  {name}: {param.shape}")

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for data, target in loader: # 每次从 loader 中拿到的样本数通常等于 batch_size，但最后一个批次可能更少。
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.size(0)  # 累加批次损失，乘以批次大小以计算加权平均
        pred = output.argmax(dim=1)  # 获取预测类别（取概率最大的类别索引）
        correct += pred.eq(target).sum().item()  # 统计预测正确的样本数，0-dim tensor通过 item() 方法转换为 float64 类型
        total += data.size(0)  # 统计总样本数
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = criterion(output, target)
        total_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += data.size(0)
    return total_loss / total, correct / total

In [ ]:
model = MLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

epochs = 10
for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    scheduler.step()  # 每个 epoch 结束时更新学习率

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(f"Epoch {epoch+1:2d}/{epochs} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history["train_loss"], "b-", label="Train")
ax1.plot(history["test_loss"], "r-", label="Test")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Curves")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history["train_acc"], "b-", label="Train")
ax2.plot(history["test_acc"], "r-", label="Test")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy Curves")
# ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0.97, color="green", linestyle="--", alpha=0.5, label="97%")
ax2.legend()

plt.suptitle(f"MLP on MNIST (Final Test Acc: {history['test_acc'][-1]:.2%})", fontsize=13)
plt.tight_layout()
plt.show()

## 2.9 单元小结

| 概念 | 要点 |
|------|------|
| **nn.Module** | 所有网络的基类，管理参数和前向传播 |
| **nn.Linear** | $y = xW^T + b$，全连接层 |
| **激活函数** | ReLU (首选), Sigmoid, Tanh, Softmax |
| **损失函数** | MSELoss (回归), CrossEntropyLoss (多分类), BCEWithLogitsLoss (二分类) |
| **优化器** | SGD (需调参), Adam/AdamW (推荐) |
| **训练循环** | `zero_grad() -> loss.backward() -> optimizer.step()` |

### 思考题
1. 为什么 `CrossEntropyLoss` 不需要手动添加 Softmax？
2. `model.train()` 和 `model.eval()` 分别做了什么？
3. 如果把 ReLU 全部换成 Sigmoid 会怎样？（提示：梯度消失）